# <font color = 'indianred'>**Multilabel Emotion Detection using GEMMA (Instruction Model)** </font>

**Objective:**  
Fine-tune an *instruction-tuned decoder model* (Gemma-2-2B-IT) using QLoRA to perform **multi-label emotion detection**.

This notebook mirrors the provided reference notebook, adapted for HW6:
1. Build a chat-style instruction dataset (user asks to classify; assistant outputs labels).
2. Fine-tune Gemma with a language head to **generate labels**.
3. Track experiments with Weights & Biases (WandB).
4. Save/export the best LoRA adapter for Part C inference with vLLM.


# <font color = 'indianred'> **1. Setting up the Environment** </font>



In [1]:
!nvidia-smi

Sat Dec 13 07:26:34 2025       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 550.54.15              Driver Version: 550.54.15      CUDA Version: 12.4     |
|-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA A100-SXM4-80GB          Off |   00000000:00:05.0 Off |                    0 |
| N/A   32C    P0             53W /  400W |       0MiB /  81920MiB |      0%      Default |
|                                         |                        |             Disabled |
+-----------------------------------------+-----

In [2]:
# If in Colab, then import the drive module from google.colab
if 'google.colab' in str(get_ipython()):
  # !pip install uv
  !pip install numpy -U -qq
  !pip install transformers evaluate wandb datasets accelerate trl peft bitsandbytes -U -qq
  !pip uninstall tensorflow -y -qq

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.1/62.1 kB 4.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.6/16.6 MB 112.7 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
opencv-python 4.12.0.88 requires numpy<2.3.0,>=2; python_version >= "3.9", but you have numpy 2.3.5 which is incompatible.
numba 0.60.0 requires numpy<2.1,>=1.22, but you have numpy 2.3.5 which is incompatible.
opencv-python-headless 4.12.0.88 requires numpy<2.3.0,>=2; python_version >= "3.9", but you have numpy 2.3.5 which is incompatible.
opencv-contrib-python 4.12.0.88 requires numpy<2.3.0,>=2; python_version >= "3.9", but you have numpy 2.3.5 which is incompatible.
tensorflow 2.19.0 requires numpy<2.2.0,>=1.26.0, but you have numpy 2.3.5 which is incompatible.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 5.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━

 <Font size = 5 color = 'indianred'>**Restart the session before moving onto next cell**
> Runtime- Restart Session

<font color = 'indianred'> *Load Libraries* </font>

In [1]:
# standard python libraries
from pathlib import Path
from typing import Dict, List, Union, Optional, Tuple
from tqdm import tqdm
import json
import joblib
import os
import sys

# Data Science librraies
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.metrics import multilabel_confusion_matrix, precision_score, recall_score, f1_score

# Pytorch
import torch
import torch.nn as nn

# Huggingface Librraies
import evaluate
from datasets import load_dataset, DatasetDict, Dataset, ClassLabel
from trl import SFTConfig, SFTTrainer
from transformers import (
    set_seed,
    AutoTokenizer,
    AutoModelForCausalLM,
    DataCollatorForLanguageModeling,
    AutoConfig,
    BitsAndBytesConfig,
)
from peft import (
    TaskType,
    LoraConfig,
    prepare_model_for_kbit_training,
    get_peft_model,
    AutoPeftModelForCausalLM,
    PeftConfig
)
# Logging and secrets
from huggingface_hub import login, HfApi, create_repo
from google.colab import userdata
import wandb



In [2]:
sys.path

['/content',
 '/env/python',
 '/usr/lib/python312.zip',
 '/usr/lib/python3.12',
 '/usr/lib/python3.12/lib-dynload',
 '',
 '/usr/local/lib/python3.12/dist-packages',
 '/usr/lib/python3/dist-packages',
 '/usr/local/lib/python3.12/dist-packages/IPython/extensions',
 '/root/.ipython',
 '/tmp/tmpbb50_wgx']

In [3]:
from pathlib import Path
# Determine the storage location based on the execution environment
# If running on Google Colab, use Google Drive as storage
if 'google.colab' in str(get_ipython()):
    from google.colab import drive  # Import Google Drive mounting utility
    drive.mount('/content/drive')  # Mount Google Drive

    # Set base folder path for storing data on Google Drive
    base_folder= Path('/content/drive/My Drive/Fall_2025_Notes/NLP/HW 6')
    project_folder = Path('/content/drive/My Drive/Fall_2025_Notes/NLP/HW 6')
# If running locally, specify a different path
else:
    # Set base folder path for storing data on local machine
    base_folder= Path('/home/')
    project_folder = Path('/home/')

Mounted at /content/drive


In [4]:
#This whole block of code is the shared_utils py
import gc
import time
import torch
import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import torch.nn as nn

from sklearn.metrics import multilabel_confusion_matrix, precision_score, recall_score, f1_score
def free_gpu_memory():
    """
    Frees up GPU memory after CUDA out-of-memory error in Colab.

    This function performs the following steps:
    1. Deletes all PyTorch objects to clear references.
    2. Calls garbage collection to remove unreferenced objects from memory.
    3. Uses torch.cuda.empty_cache() to release cached GPU memory.
    4. Waits for a moment to ensure memory is fully released.
    """
    try:
        # Delete all torch tensors to free up memory
        for obj in list(locals().values()):
            if torch.is_tensor(obj):
                del obj

        # Collect garbage to release any remaining unused memory
        gc.collect()

        # Empty the CUDA cache to release GPU memory
        torch.cuda.empty_cache()

        # Adding a small delay to allow memory to be fully released
        time.sleep(2)

        print("GPU memory has been freed.")
    except Exception as e:
        print(f"Error while freeing GPU memory: {e}")

def find_linear_layers(model):
    """
    Extracts the unique names of Linear layers from a model.

    Args:
    model (nn.Module): The model from which to extract Linear layer names.

    Returns:
    list: A list of unique names of Linear layers.
    """
    # Convert the model's modules to string
    model_modules = str(model.modules)
    # Pattern to extract names of Linear layers
    pattern = r'\((\w+)\): Linear'
    # Find all occurrences of the pattern
    linear_layer_names = re.findall(pattern, model_modules)
    print(linear_layer_names)
    # Get unique names using a set, then convert back to list
    target_modules = list(set(linear_layer_names))
    return target_modules

def verify_loss_masking_detailed(tokenizer,trainer, num_samples=3):
    """
    Verify that labels are properly masked for prompt tokens.
    Shows exact tokens where loss is calculated.
    """
    # Get a sample batch
    dataloader = trainer.get_train_dataloader()
    batch = next(iter(dataloader))

    for i in range(min(num_samples, len(batch['input_ids']))):
        input_ids = batch['input_ids'][i]
        labels = batch['labels'][i]

        print(f"\n{'='*80}")
        print(f"Sample {i+1}:")
        print(f"{'='*80}")

        # Decode the full sequence
        full_text = tokenizer.decode(input_ids, skip_special_tokens=False)
        print(f"\nFull text:\n{full_text}")

        # Find where labels start (not -100)
        label_start_idx = (labels != -100).nonzero(as_tuple=True)[0]

        if len(label_start_idx) > 0:
            first_label_idx = label_start_idx[0].item()

            # Decode prompt (masked part)
            prompt_ids = input_ids[:first_label_idx]
            prompt_text = tokenizer.decode(prompt_ids, skip_special_tokens=False)

            # Decode completion (unmasked part)
            completion_ids = input_ids[first_label_idx:]
            completion_text = tokenizer.decode(completion_ids, skip_special_tokens=False)

            print(f"\n{'─'*80}")
            print(f"PROMPT (masked with -100):\n{prompt_text}")
            print(f"\n{'─'*80}")
            print(f"COMPLETION (loss calculated here):\n{completion_text}")
            print(f"\n{'─'*80}")

            # Show label statistics
            num_masked = (labels == -100).sum().item()
            num_unmasked = (labels != -100).sum().item()
            print(f"\nLabel Statistics:")
            print(f"  Masked tokens (prompt): {num_masked}")
            print(f"  Unmasked tokens (completion): {num_unmasked}")
            print(f"  Total tokens: {len(labels)}")
            print(f"  Percentage of tokens used for loss: {num_unmasked/len(labels)*100:.2f}%")

            # ===== NEW: Show exact tokens where loss is calculated =====
            print(f"\n{'='*80}")
            print("EXACT TOKENS WHERE LOSS IS CALCULATED:")
            print(f"{'='*80}")
            print(f"{'Index':<8} {'Token ID':<12} {'Token':<40} {'Label ID':<12}")
            print(f"{'-'*80}")

            for idx in range(len(input_ids)):
                token_id = input_ids[idx].item()
                label_id = labels[idx].item()

                # Only show tokens where loss is calculated (label != -100)
                if label_id != -100:
                    # Decode individual token
                    token_text = tokenizer.decode([token_id], skip_special_tokens=False)
                    # Show what the model should predict (the label)
                    label_text = tokenizer.decode([label_id], skip_special_tokens=False)

                    print(f"{idx:<8} {token_id:<12} {repr(token_text):<40} {label_id:<12}")

            print(f"{'-'*80}")

            # Additional: Show first few masked tokens for comparison
            print(f"\n{'='*80}")
            print("FIRST 10 MASKED TOKENS (for comparison - NO loss calculated):")
            print(f"{'='*80}")
            print(f"{'Index':<8} {'Token ID':<12} {'Token':<40} {'Label':<12}")
            print(f"{'-'*80}")

            masked_count = 0
            for idx in range(len(input_ids)):
                token_id = input_ids[idx].item()
                label_id = labels[idx].item()

                if label_id == -100:
                    token_text = tokenizer.decode([token_id], skip_special_tokens=False)
                    print(f"{idx:<8} {token_id:<12} {repr(token_text):<40} {'-100':<12}")
                    masked_count += 1
                    if masked_count >= 10:
                        break

            print(f"{'-'*80}")

        else:
            print("WARNING: All labels are masked! No loss will be calculated.")

    print(f"\n{'='*80}\n")

def multilabel_evaluation(y_true, y_pred, class_names=None, figsize=(12, 8)):
    """
    Generate comprehensive evaluation visualizations for multilabel classification results.

    Parameters:
    -----------
    y_true : array-like
        True labels (n_samples, n_classes)
    y_pred : array-like
        Predicted labels (n_samples, n_classes)
    class_names : list, optional
        List of class names for better visualization
    figsize : tuple, optional
        Base figure size for plots (width, height)

    Returns:
    --------
    dict
        Dictionary containing the computed metrics for each class
    """
    # Validate inputs
    y_true = np.array(y_true)
    y_pred = np.array(y_pred)

    if y_true.shape != y_pred.shape:
        raise ValueError("y_true and y_pred must have the same shape")

    # Generate class names if not provided
    if class_names is None:
        class_names = [f'Class {i}' for i in range(y_true.shape[1])]

    # Calculate confusion matrices
    mcm = multilabel_confusion_matrix(y_true, y_pred)

    # 1. Individual Confusion Matrix Heatmaps
    n_classes = len(class_names)
    n_cols = min(3, n_classes)
    n_rows = (n_classes + n_cols - 1) // n_cols

    plt.figure(figsize=(figsize[0], figsize[1] * n_rows/2))
    for idx, matrix in enumerate(mcm):
        plt.subplot(n_rows, n_cols, idx + 1)
        sns.heatmap(matrix, annot=True, fmt='g', cmap='Blues',
                    xticklabels=['Pred Neg', 'Pred Pos'],
                    yticklabels=['True Neg', 'True Pos'])
        plt.title(f'{class_names[idx]}')
    plt.tight_layout()
    plt.show()

    # 2. Calculate and plot aggregate metrics
    metrics = {
        'Precision': precision_score(y_true, y_pred, average=None),
        'Recall': recall_score(y_true, y_pred, average=None),
        'F1-Score': f1_score(y_true, y_pred, average=None)
    }

    metrics_df = pd.DataFrame(metrics, index=class_names)

    # Metrics Heatmap
    plt.figure(figsize=(figsize[0]/1.5, figsize[1]/1.5))
    sns.heatmap(metrics_df, annot=True, fmt='.3f', cmap='Blues')
    plt.title('Performance Metrics by Class')
    plt.tight_layout()
    plt.show()

    # 3. Metrics Histogram
    plt.figure(figsize=(figsize[0], figsize[1]/1.5))
    metrics_df.plot(kind='bar', width=0.8)
    plt.xlabel('Classes')
    plt.ylabel('Score')
    plt.title('Precision, Recall, and F1-Score by Class')
    plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left')
    plt.tight_layout()
    plt.show()

    # 4. Calculate and return summary statistics
    summary_stats = {
        'macro_avg': {
            'precision': np.mean(metrics['Precision']),
            'recall': np.mean(metrics['Recall']),
            'f1': np.mean(metrics['F1-Score'])
        },
        'per_class': metrics_df.to_dict()
    }

    return summary_stats

def get_appropriate_dtype():
    if torch.cuda.is_available() and torch.cuda.get_device_capability(0) >= (8, 0):
        return torch.bfloat16
    return torch.float16

In [6]:
wandb_api_key = userdata.get('WANDB_API_KEY')
hf_token = userdata.get('HF_TOKEN')

In [7]:
if hf_token:
    # Log in to Hugging Face
    login(token=hf_token)
    print("Successfully logged in to Hugging Face!")
else:
    print("Hugging Face token not found in notebook secrets.")

Successfully logged in to Hugging Face!


In [8]:
if wandb_api_key:
  wandb.login(key=wandb_api_key)
  print("Successfully logged in to WANDB!")
else:
    print("WANDB key not found in notebook secrets.")

/usr/local/lib/python3.12/dist-packages/notebook/notebookapp.py:191: SyntaxWarning: invalid escape sequence '\/'
  | |_| | '_ \/ _` / _` |  _/ -_)
wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: stevenqcly (stevenqcly-the-university-of-texas-at-dallas) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


Successfully logged in to WANDB!


# <font color = 'indianred'> **2. Load Data set**
    


In [214]:
# Students can set the data_folder and model_folder paths based on their base_folder setup.
data_folder = base_folder/'datasets'
model_folder = project_folder/'models'
model_folder.mkdir(exist_ok=True, parents = True)
data_folder.mkdir(exist_ok=True, parents = True)

# Load train/test CSVs
train_path = data_folder / "train.csv"
test_path  = data_folder / "test.csv"

train_df = pd.read_csv(train_path)
kaggle_test_df = pd.read_csv(test_path)

TEXT_COL = "Tweet"
ID_COL = "ID"

label_cols = [c for c in train_df.columns if c not in [ID_COL, TEXT_COL]]
label_cols


['anger',
 'anticipation',
 'disgust',
 'fear',
 'joy',
 'love',
 'optimism',
 'pessimism',
 'sadness',
 'surprise',
 'trust']

In [215]:
# Ensure label columns are numeric
train_df[label_cols] = train_df[label_cols].apply(pd.to_numeric, errors="coerce").fillna(0).astype(int)

def build_label_string_from_row(row) -> str:
    labels = [lbl for lbl in label_cols if row[lbl] == 1]
    return json.dumps(labels)

df = train_df[[TEXT_COL] + label_cols].copy()
df["label"] = df.apply(build_label_string_from_row, axis=1)
df = df.rename(columns={TEXT_COL: "text"})
df = df[["text", "label"]]
df.head()


,text,label
0,“Worry is a down payment on a problem you may ...,"[""anticipation"", ""optimism"", ""trust""]"
1,Whatever you decide to do make sure it makes y...,"[""joy"", ""love"", ""optimism""]"
2,@Max_Kellerman it also helps that the majorit...,"[""anger"", ""disgust"", ""joy"", ""optimism""]"
3,Accept the challenges so that you can literall...,"[""joy"", ""optimism""]"
4,My roommate: it's okay that we can't spell bec...,"[""anger"", ""disgust""]"


In [216]:
# Convert to HF Dataset
stack_selected_columns_final = Dataset.from_pandas(df)
stack_selected_columns_final[0]


{'text': "“Worry is a down payment on a problem you may never have'. \xa0Joyce Meyer.  #motivation #leadership #worry",
 'label': '["anticipation", "optimism", "trust"]'}

In [217]:
stack_selected_columns_final['label'][0]


'["anticipation", "optimism", "trust"]'

In [218]:
stack_selected_columns_final['label'][0]

'["anticipation", "optimism", "trust"]'

In [219]:
stack_selected_columns = stack_selected_columns_final

# <font color = 'indianred'> **3. Accessing and Manuplating Splits**</font>



<font color = 'indianred'>*Create futher subdivions of the splits*</font>

In [220]:
# Split into train/valid/test (matches earlier parts)
SEED = 42

# First split: hold out test (15%)
train_val_splits = stack_selected_columns.train_test_split(test_size=0.15, seed=SEED, shuffle=True)
train_val = train_val_splits['train']
test_split = train_val_splits['test']

# Second split: validation from remaining (~15% of total)
tv_splits = train_val.train_test_split(test_size=0.15/0.85, seed=SEED, shuffle=True)
train_split = tv_splits['train']
val_split   = tv_splits['test']

train_split, val_split, test_split


(Dataset({
     features: ['text', 'label'],
     num_rows: 5406
 }),
 Dataset({
     features: ['text', 'label'],
     num_rows: 1159
 }),
 Dataset({
     features: ['text', 'label'],
     num_rows: 1159
 }))

<font color = 'indianred'>*small subset for initial experimenttaion*</font>

In [128]:
# Optional: small subset for quick experimentation (uncomment if needed)
# train_split = train_split.shuffle(seed=42).select(range(min(4000, len(train_split))))
# val_split   = val_split.shuffle(seed=42).select(range(min(1000, len(val_split))))
# test_split  = test_split.shuffle(seed=42).select(range(min(1000, len(test_split))))


In [221]:
data_subset = DatasetDict({"train": train_split, "valid": val_split, "test": test_split})
data_subset


DatasetDict({
    train: Dataset({
        features: ['text', 'label'],
        num_rows: 5406
    })
    valid: Dataset({
        features: ['text', 'label'],
        num_rows: 1159
    })
    test: Dataset({
        features: ['text', 'label'],
        num_rows: 1159
    })
})

In [222]:
data_subset

DatasetDict({
    train: Dataset({
        features: ['text', 'label'],
        num_rows: 5406
    })
    valid: Dataset({
        features: ['text', 'label'],
        num_rows: 1159
    })
    test: Dataset({
        features: ['text', 'label'],
        num_rows: 1159
    })
})

In [223]:
data_subset['train'][0]


{'text': "Won't be watching ESPN anymore, they took Robert Lee off because of HIS name, PC Police, what insanity! more #hatred #bigots #LeftwingNutJob",
 'label': '["anger", "disgust"]'}

# <font color = 'indianred'>**4. Load pre-trained Tokenizer**</font>

In [224]:
free_gpu_memory()

GPU memory has been freed.


In [225]:
free_gpu_memory()

checkpoint = "google/gemma-2-2b-it"
tokenizer = AutoTokenizer.from_pretrained(checkpoint)
tokenizer


GPU memory has been freed.


GemmaTokenizerFast(name_or_path='google/gemma-2-2b-it', vocab_size=256000, model_max_length=1000000000000000019884624838656, is_fast=True, padding_side='left', truncation_side='right', special_tokens={'bos_token': '<bos>', 'eos_token': '<eos>', 'unk_token': '<unk>', 'pad_token': '<pad>', 'additional_special_tokens': ['<start_of_turn>', '<end_of_turn>']}, clean_up_tokenization_spaces=False, added_tokens_decoder={
	0: AddedToken("<pad>", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	1: AddedToken("<eos>", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	2: AddedToken("<bos>", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	3: AddedToken("<unk>", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	4: AddedToken("<mask>", rstrip=False, lstrip=False, single_word=False, normalized=False, special=False),
	5: AddedToken("<2mass>", rstrip=False, lstrip=False, single_w

In [226]:
tokenizer.eos_token

'<eos>'

In [227]:
tokenizer.pad_token

'<pad>'

In [228]:
tokenizer.padding_side

'left'

In [229]:
from pprint import pprint
pprint(tokenizer.chat_template)

("{{ bos_token }}{% if messages[0]['role'] == 'system' %}{{ "
 "raise_exception('System role not supported') }}{% endif %}{% for message in "
 "messages %}{% if (message['role'] == 'user') != (loop.index0 % 2 == 0) %}{{ "
 "raise_exception('Conversation roles must alternate "
 "user/assistant/user/assistant/...') }}{% endif %}{% if (message['role'] == "
 "'assistant') %}{% set role = 'model' %}{% else %}{% set role = "
 "message['role'] %}{% endif %}{{ '<start_of_turn>' + role + '\n"
 "' + message['content'] | trim + '<end_of_turn>\n"
 "' }}{% endfor %}{% if add_generation_prompt %}{{'<start_of_turn>model\n"
 "'}}{% endif %}")


In [230]:
# 2. Test the template works
print("Testing custom chat template...")
test_result = tokenizer.apply_chat_template(
    [{"role": "user", "content": "Test"}, {"role": "assistant", "content": "Response"}],
    return_dict=True,
    return_assistant_tokens_mask=True,
    add_generation_prompt=False
)
print(test_result)
assistant_tokens = sum(test_result.get('assistant_masks', []))
print(f"Assistant tokens detected: {assistant_tokens}")

Testing custom chat template...
{'input_ids': [2, 106, 1645, 108, 2015, 107, 108, 106, 2516, 108, 3943, 107, 108], 'attention_mask': [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1], 'assistant_masks': [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]}
Assistant tokens detected: 0


In [231]:
# Modified template to only train on the actual content
tokenizer.chat_template = """
{{ bos_token }}
{% if messages[0]['role'] == 'system' %}
  {{ raise_exception('System role not supported') }}
{% endif %}
{% for message in messages %}
  {% if (message['role'] == 'user') != (loop.index0 % 2 == 0) %}
    {{ raise_exception('Conversation roles must alternate user/assistant/user/assistant/...') }}
  {% endif %}
  {% if message['role'] == 'assistant' %}
<start_of_turn>model
{% generation %}{{ message['content'] | trim }}
<end_of_turn>{% endgeneration %}
  {% else %}
<start_of_turn>{{ message['role'] }}
{{ message['content'] | trim }}
<end_of_turn>
  {% endif %}
{% endfor %}
{% if add_generation_prompt %}
<start_of_turn>model
{% endif %}
"""


In [232]:
# 2. Test the template works
print("Testing custom chat template...")
test_result = tokenizer.apply_chat_template(
    [{"role": "user", "content": "Test"}, {"role": "assistant", "content": "Response"}],
    return_dict=True,
    return_assistant_tokens_mask=True,
    add_generation_prompt=False
)
print(test_result)
assistant_tokens = sum(test_result.get('assistant_masks', []))
print(f"Assistant tokens detected: {assistant_tokens}")

Testing custom chat template...
{'input_ids': [108, 2, 108, 106, 1645, 108, 2015, 108, 107, 108, 106, 2516, 108, 3943, 108, 107], 'attention_mask': [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1], 'assistant_masks': [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 1, 1]}
Assistant tokens detected: 3


In [233]:
# Decode individual tokens to see what they are
print("\nToken-by-token breakdown:")
print("Index | Token ID | Token Text | Assistant?")
print("-" * 50)
input_ids = test_result['input_ids']
assistant_masks = test_result['assistant_masks']

for i, (token_id, is_assistant) in enumerate(zip(input_ids, assistant_masks)):
    token_text = tokenizer.decode([token_id])
    status = "ASSISTANT" if is_assistant else "Other"
    print(f"{i:5d} | {token_id:8d} | {repr(token_text):15s} | {status}")

print(f"\nFull text: {repr(tokenizer.decode(input_ids))}")

# Show only assistant tokens
assistant_token_ids = [token_id for token_id, is_assistant in zip(input_ids, assistant_masks) if is_assistant]
assistant_only_text = tokenizer.decode(assistant_token_ids)
print(f"Assistant-only text: {repr(assistant_only_text)}")

# Show the masked (non-assistant) tokens
user_token_ids = [token_id for token_id, is_assistant in zip(input_ids, assistant_masks) if not is_assistant]
user_only_text = tokenizer.decode(user_token_ids)
print(f"User/Other text: {repr(user_only_text)}")


Token-by-token breakdown:
Index | Token ID | Token Text | Assistant?
--------------------------------------------------
    0 |      108 | '\n'            | Other
    1 |        2 | '<bos>'         | Other
    2 |      108 | '\n'            | Other
    3 |      106 | '<start_of_turn>' | Other
    4 |     1645 | 'user'          | Other
    5 |      108 | '\n'            | Other
    6 |     2015 | 'Test'          | Other
    7 |      108 | '\n'            | Other
    8 |      107 | '<end_of_turn>' | Other
    9 |      108 | '\n'            | Other
   10 |      106 | '<start_of_turn>' | Other
   11 |     2516 | 'model'         | Other
   12 |      108 | '\n'            | Other
   13 |     3943 | 'Response'      | ASSISTANT
   14 |      108 | '\n'            | ASSISTANT
   15 |      107 | '<end_of_turn>' | ASSISTANT

Full text: '\n<bos>\n<start_of_turn>user\nTest\n<end_of_turn>\n<start_of_turn>model\nResponse\n<end_of_turn>'
Assistant-only text: 'Response\n<end_of_turn>'
User/Other text: 

#<font color = 'indianred'> **5. Create Chat Dataset**



In [234]:
class_names = label_cols

In [91]:
# class_names

# def format_chat(example):
#     instruction = (
#         "You are an expert emotion classifier. "
#         f"Given the TEXT, output ONLY a valid JSON array of emotion labels chosen from: {class_names}. "
#         "Do not include any explanation."
#         f"\n\nTEXT: {example['text']}"
#     )
#     messages = [
#         {"role": "user", "content": instruction},
#         {"role": "assistant", "content": example["label"]},
#     ]
#     return {"messages": messages}

# data_subset_chat = data_subset.map(format_chat, remove_columns=["text", "label"])
# data_subset_chat


In [92]:
#BACKUP
# data_subset_chat = data_subset.map(format_chat, remove_columns=["text", "label"])

In [235]:
def format_chat_example(example):
    user_msg = (
        "You are an expert emotion classifier. Given the TEXT, output ONLY a valid "
        f"JSON array of emotion labels chosen from: {label_cols}. Do not include any explanation.\n\n"
        f"TEXT: {example['text'].strip()}"
    )
    assistant_msg = example["label"].strip()  # already a JSON string like ["joy","love"]

    messages = [
        {"role": "user", "content": user_msg},
        {"role": "assistant", "content": assistant_msg},
    ]

    # Full text used for training
    full_text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=False)

    # Prompt-only text (everything before assistant content)
    prompt_messages = [{"role": "user", "content": user_msg}]
    prompt_text = tokenizer.apply_chat_template(prompt_messages, tokenize=False, add_generation_prompt=True)

    # Token lengths
    prompt_len = len(tokenizer(prompt_text, add_special_tokens=False).input_ids)

    return {"text": full_text, "prompt_len": prompt_len}

In [236]:
data_subset = data_subset.map(
    format_chat_example,
    remove_columns=data_subset["train"].column_names
)

print(data_subset["train"].column_names)
print(data_subset["train"][0].keys())

Map:   0%|          | 0/5406 [00:00<?, ? examples/s]

Map:   0%|          | 0/1159 [00:00<?, ? examples/s]

Map:   0%|          | 0/1159 [00:00<?, ? examples/s]

['text', 'prompt_len']
dict_keys(['text', 'prompt_len'])


##  <font color = 'indianred'> **5.1 Filter Longer sequences**

In [208]:
# def check_length(example):
#     # Find the first user message
#     text_to_check = None
#     for msg in example.get("messages", []):
#         if msg.get("role") == "user":
#             text_to_check = msg.get("content", "").strip()
#             break

#     # Return False if there's no user message or empty text
#     if not text_to_check:
#         return False

#     # Tokenize the text and check length
#     encoding = tokenizer.encode(text_to_check)
#     return len(encoding) <= 500


In [237]:
def keep_short(example):
    return example["prompt_len"] <= 500

filtered_data_subset_chat = DatasetDict({
    split: data_subset[split].filter(keep_short)
    for split in data_subset.keys()
})

print(filtered_data_subset_chat["train"].column_names)
print(filtered_data_subset_chat["train"][0].keys())

Filter:   0%|          | 0/5406 [00:00<?, ? examples/s]

Filter:   0%|          | 0/1159 [00:00<?, ? examples/s]

Filter:   0%|          | 0/1159 [00:00<?, ? examples/s]

['text', 'prompt_len']
dict_keys(['text', 'prompt_len'])


##  <font color = 'indianred'> **5.2 Push Dataset to Hub**

In [167]:
# OPTIONAL: push dataset to Hub is this required? I'm going to comment this block and go back to look at this later!
# filtered_data_subset_chat.push_to_hub(
#     "<YOUR_HF_USERNAME>/hw6_emotions_partc_chat",
#     private=False
# )


#  <font color = 'indianred'> **6. Model Training**

##  <font color = 'indianred'> **6.1 Download pre-trained model**

In [238]:
def get_appropriate_dtype():
    if torch.cuda.is_available() and torch.cuda.get_device_capability(0) >= (8, 0):
        return torch.bfloat16
    return torch.float16

In [239]:
torch_data_type = get_appropriate_dtype()
torch_data_type

torch.bfloat16

In [240]:
bnb_config = BitsAndBytesConfig(
  load_in_4bit=True,
  bnb_4bit_quant_type="nf4",
  bnb_4bit_use_double_quant=True,
  bnb_4bit_compute_dtype=torch_data_type,
  bnb_4bit_quant_storage=torch_data_type,
)

In [241]:
model = AutoModelForCausalLM.from_pretrained(checkpoint,
                                             quantization_config=bnb_config,
                                             torch_dtype=torch_data_type,
                                             trust_remote_code=True,)


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

##  <font color = 'indianred'> **6.2 PEFT Setup**

In [242]:
model

Gemma2ForCausalLM(
  (model): Gemma2Model(
    (embed_tokens): Embedding(256000, 2304, padding_idx=0)
    (layers): ModuleList(
      (0-25): 26 x Gemma2DecoderLayer(
        (self_attn): Gemma2Attention(
          (q_proj): Linear4bit(in_features=2304, out_features=2048, bias=False)
          (k_proj): Linear4bit(in_features=2304, out_features=1024, bias=False)
          (v_proj): Linear4bit(in_features=2304, out_features=1024, bias=False)
          (o_proj): Linear4bit(in_features=2048, out_features=2304, bias=False)
        )
        (mlp): Gemma2MLP(
          (gate_proj): Linear4bit(in_features=2304, out_features=9216, bias=False)
          (up_proj): Linear4bit(in_features=2304, out_features=9216, bias=False)
          (down_proj): Linear4bit(in_features=9216, out_features=2304, bias=False)
          (act_fn): GELUTanh()
        )
        (input_layernorm): Gemma2RMSNorm((2304,), eps=1e-06)
        (post_attention_layernorm): Gemma2RMSNorm((2304,), eps=1e-06)
        (pre_feedfor

In [243]:
find_linear_layers(model)

['q_proj', 'k_proj', 'v_proj', 'o_proj', 'gate_proj', 'up_proj', 'down_proj', 'lm_head']


['v_proj',
 'lm_head',
 'gate_proj',
 'o_proj',
 'up_proj',
 'k_proj',
 'down_proj',
 'q_proj']

In [244]:
TaskType.CAUSAL_LM

<TaskType.CAUSAL_LM: 'CAUSAL_LM'>

In [245]:
peft_config = LoraConfig(
    task_type=TaskType.CAUSAL_LM,
    r=64,#<- halving
    lora_alpha=128,#<-also halving
    lora_dropout=0.01,
    target_modules = ['v_proj',  'q_proj',  'up_proj', 'o_proj', 'down_proj', 'gate_proj','k_proj'])



## <font color = 'indianred'> **6.3 Training Arguments**</font>







In [246]:
# Define the directory where model checkpoints will be saved
model_folder = Path("/content/models/gemma_partc_instruction_lmh") #again, i'm saving the model checkpoints to the colab environment instead of my Drive
model_folder.mkdir(exist_ok=True, parents=True)
run_name = "hw6_partc_instruction_lmh_gemma2_2b_it"


In [247]:
import os, wandb

os.environ["WANDB_PROJECT"] = "multilabel_PartC_2025"

run = wandb.init(
    project=os.environ["WANDB_PROJECT"],
    name=run_name,
)

print("W&B run:", run.url)


eval/entropy,█▃▃▃▃▃▃▂▂▁▁▁▁▁▁▁
eval/loss,█▅▄▃▃▂▂▂▁▁▁▁▁▁▁▁
eval/mean_token_accuracy,▁▄▅▅▆▆▇▇████████
eval/num_tokens,▁▁▂▂▃▃▄▄▅▅▆▆▇▇██
eval/runtime,▅█▂▁▁▃▃▃▂▃▃▃▄▂▃▃
eval/samples_per_second,▄▁▇██▆▆▆▇▆▆▆▅▇▆▆
eval/steps_per_second,▄▁▇██▆▆▆▇▆▆▆▅▇▆▆
train/entropy,█▄▃▃▂▃▂▂▂▁▁▁▁▁▁▁
train/epoch,▁▁▁▁▂▂▂▂▃▃▃▃▄▄▄▄▅▅▅▅▅▅▆▆▆▆▇▇▇▇███
train/global_step,▁▁▁▁▂▂▂▂▃▃▃▃▄▄▄▄▅▅▅▅▅▅▆▆▆▆▇▇▇▇███
+5,...


W&B run: https://wandb.ai/stevenqcly-the-university-of-texas-at-dallas/multilabel_PartC_2025/runs/g5b754n5


In [248]:
#Training Args
from trl import SFTConfig

use_fp16 = (torch_data_type == torch.float16)
use_bf16 = (torch_data_type == torch.bfloat16)

training_args = SFTConfig(
    seed=42,
    dataset_text_field="text",
    remove_unused_columns=False, #<- adding this because sft keeps dropping my columns
    max_length=1024,
    packing=False,
    completion_only_loss=True,
    group_by_length=True, #<- adding this to reduce bad padding and hopefully increase model performance

    num_train_epochs=3, #<- slightly increasing empoch for performance
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    gradient_accumulation_steps=2,
    gradient_checkpointing=True,
    gradient_checkpointing_kwargs={"use_reentrant": False},

    learning_rate=2e-5, #<- slightly increase lr for better perforamnce (my loss is plateauing when it is at 1e-5)
    weight_decay=0.0,
    optim="adamw_torch",
    warmup_ratio=0.03, #<- adding this too

    output_dir=str(model_folder),
    eval_strategy="steps",
    eval_steps=20,
    save_strategy="steps",
    save_steps=20,
    save_total_limit=2,
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,

    logging_strategy="steps",
    logging_steps=20,
    report_to="wandb",
    run_name=run_name,

    fp16=use_fp16,
    bf16=use_bf16,
    tf32=False,
)


In [249]:
# If gradient checkpointing is enabled, configure relevant settings
if training_args.gradient_checkpointing:
    model.config.use_cache = False  # Disable caching for compatibility



In [250]:
#Similar to my part B, the Datacollator from TRL is broken in my version of Colab, so the masking is not working correctly further down. I have to use this workaround masker.
from dataclasses import dataclass
from typing import Any, Dict, List
import torch

@dataclass
class DataCollatorForPromptLenCompletionOnlyLM:
    tokenizer: Any
    max_length: int = 1024

    def __call__(self, features: List[Dict[str, Any]]) -> Dict[str, torch.Tensor]:
        texts = [f["text"] for f in features]
        prompt_lens = [int(f["prompt_len"]) for f in features]

        batch = self.tokenizer(
            texts,
            padding=True,
            truncation=True,
            max_length=self.max_length,
            return_tensors="pt",
        )

        input_ids = batch["input_ids"]
        attention_mask = batch["attention_mask"]
        labels = input_ids.clone()

        for i, pl in enumerate(prompt_lens):
            pl = min(pl, labels.size(1))
            labels[i, :pl] = -100

        labels = labels.masked_fill(attention_mask == 0, -100)
        batch["labels"] = labels
        return batch


data_collator = DataCollatorForPromptLenCompletionOnlyLM(
    tokenizer=tokenizer,
    max_length=training_args.max_length
)


##  <font color = 'indianred'> **6.4 Initialize Trainer**</font>



In [251]:
print(filtered_data_subset_chat["train"][0]["prompt_len"])
print(filtered_data_subset_chat["train"][0]["text"][:300])


119

<bos>
<start_of_turn>user
You are an expert emotion classifier. Given the TEXT, output ONLY a valid JSON array of emotion labels chosen from: ['anger', 'anticipation', 'disgust', 'fear', 'joy', 'love', 'optimism', 'pessimism', 'sadness', 'surprise', 'trust']. Do not include any explanation.

TEXT: 


In [252]:
trainer = SFTTrainer(
    model=model,
    args=training_args,
    train_dataset=filtered_data_subset_chat['train'],
    eval_dataset=filtered_data_subset_chat['valid'],
    peft_config=peft_config,
    data_collator=data_collator,
    processing_class=tokenizer,
)

Adding EOS to train dataset:   0%|          | 0/5406 [00:00<?, ? examples/s]

Tokenizing train dataset:   0%|          | 0/5406 [00:00<?, ? examples/s]

Truncating train dataset:   0%|          | 0/5406 [00:00<?, ? examples/s]

Adding EOS to eval dataset:   0%|          | 0/1159 [00:00<?, ? examples/s]

Tokenizing eval dataset:   0%|          | 0/1159 [00:00<?, ? examples/s]

Truncating eval dataset:   0%|          | 0/1159 [00:00<?, ? examples/s]

In [253]:
print(filtered_data_subset_chat["train"].column_names)
print(filtered_data_subset_chat["train"][0].keys())

['text', 'prompt_len']
dict_keys(['text', 'prompt_len'])


In [254]:
dataloader = trainer.get_train_dataloader()
batch = next(iter(dataloader))

In [255]:
batch

{'input_ids': tensor([[  2, 108,   2,  ..., 108, 107,   1],
        [  0,   0,   0,  ..., 108, 107,   1],
        [  0,   0,   0,  ..., 108, 107,   1],
        ...,
        [  0,   0,   0,  ..., 108, 107,   1],
        [  0,   0,   0,  ..., 108, 107,   1],
        [  0,   0,   0,  ..., 108, 107,   1]], device='cuda:0'), 'attention_mask': tensor([[1, 1, 1,  ..., 1, 1, 1],
        [0, 0, 0,  ..., 1, 1, 1],
        [0, 0, 0,  ..., 1, 1, 1],
        ...,
        [0, 0, 0,  ..., 1, 1, 1],
        [0, 0, 0,  ..., 1, 1, 1],
        [0, 0, 0,  ..., 1, 1, 1]], device='cuda:0'), 'labels': tensor([[-100, -100, -100,  ...,  108,  107,    1],
        [-100, -100, -100,  ...,  108,  107,    1],
        [-100, -100, -100,  ...,  108,  107,    1],
        ...,
        [-100, -100, -100,  ...,  108,  107,    1],
        [-100, -100, -100,  ...,  108,  107,    1],
        [-100, -100, -100,  ...,  108,  107,    1]], device='cuda:0')}

In [256]:
print(batch['input_ids'][0][0:5])
print(tokenizer.decode(batch['input_ids'][0][0:5]))
print(batch['labels'][0][0:5])

tensor([  2, 108,   2, 108, 106], device='cuda:0')
<bos>
<bos>
<start_of_turn>
tensor([-100, -100, -100, -100, -100], device='cuda:0')


In [257]:
print(len(batch['input_ids'][0]))
print(len(batch['labels'][0]))

160
160


In [258]:
print(batch['input_ids'][0][0:5])
print(tokenizer.decode(batch['input_ids'][0][0:5]))
print(batch['labels'][0][0:5])

tensor([  2, 108,   2, 108, 106], device='cuda:0')
<bos>
<bos>
<start_of_turn>
tensor([-100, -100, -100, -100, -100], device='cuda:0')


In [259]:
print(f"\nINPUTS")
print(f"{'-'*80}")
print(batch['input_ids'][0][106:120])
print(f"\nLABELS")
print(f"{'-'*80}")
print(batch['labels'][0][106:120])
print(f"\nTokens")
print(f"{'-'*80}")
print(tokenizer.decode(batch['input_ids'][0][106:120]))


INPUTS
--------------------------------------------------------------------------------
tensor([235269, 235284, 235324, 235315,    823,  21226,    591, 235274, 235269,
        235274, 235318, 235321, 235275,    724], device='cuda:0')

LABELS
--------------------------------------------------------------------------------
tensor([-100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100,
        -100, -100], device='cuda:0')

Tokens
--------------------------------------------------------------------------------
,279), Walker (1,168) &


In [260]:
def verify_loss_masking(tokenizer, trainer, num_samples=3):
    """
    Verify which tokens contribute to loss (labels != -100)
    for a few samples from the training dataloader.
    """
    dataloader = trainer.get_train_dataloader()
    batch = next(iter(dataloader))

    for i in range(min(num_samples, len(batch["input_ids"]))):
        input_ids = batch["input_ids"][i]
        labels = batch["labels"][i]

        print(f"\n{'='*80}")
        print(f"Sample {i+1}")
        print(f"{'='*80}")

        # Decode full sequence for reference
        full_text = tokenizer.decode(input_ids, skip_special_tokens=False)
        pprint(f"\nFull text:\n{full_text}")

        # Identify tokens used for loss
        loss_token_indices = (labels != -100).nonzero(as_tuple=True)[0]

        if len(loss_token_indices) == 0:
            print("All tokens masked — no loss will be calculated.")
            continue

        print(f"\nTokens contributing to loss ({len(loss_token_indices)} total):")
        print(f"{'-'*80}")
        print(f"{'Index':<8} {'Token ID':<10} {'Token Text'}")
        print(f"{'-'*80}")

        for idx in loss_token_indices.tolist():
            token_id = input_ids[idx].item()
            token_text = tokenizer.decode([token_id], skip_special_tokens=False)
            print(f"{idx:<8} {token_id:<10} {repr(token_text)}")

        print(f"{'-'*80}")
        print(f"Percentage of tokens used for loss: {len(loss_token_indices)/len(labels)*100:.2f}%")

In [261]:
# Call this after creating your trainer
verify_loss_masking(tokenizer, trainer, num_samples=2)



Sample 1
('\n'
 'Full text:\n'
 '<bos>\n'
 '<bos>\n'
 '<start_of_turn>user\n'
 'You are an expert emotion classifier. Given the TEXT, output ONLY a valid '
 "JSON array of emotion labels chosen from: ['anger', 'anticipation', "
 "'disgust', 'fear', 'joy', 'love', 'optimism', 'pessimism', 'sadness', "
 "'surprise', 'trust']. Do not include any explanation.\n"
 '\n'
 'TEXT: .@REDBLACKS’ Chris Williams: 5 yards shy of joining 1,000-yard club. '
 'Bowman (1,279), Walker (1,168) &amp; Roosevelt (1,095) already there. #CFL\n'
 '<end_of_turn>\n'
 '<start_of_turn>model\n'
 '["anticipation", "joy", "optimism", "sadness"]\n'
 '<end_of_turn><eos>')

Tokens contributing to loss (19 total):
--------------------------------------------------------------------------------
Index    Token ID   Token Text
--------------------------------------------------------------------------------
141      108        '\n'
142      3681       '["'
143      2236       'anti'
144      76547      'cipation'
145      82

In [262]:
idx = (batch["labels"][0] != -100).nonzero(as_tuple=True)[0]
print("loss tokens:", len(idx))
print(tokenizer.decode(batch["input_ids"][0][idx[0]:idx[0]+80]))


loss tokens: 19

["anticipation", "joy", "optimism", "sadness"]
<end_of_turn><eos>


## <font color = 'indianred'> **6.5 Setup WandB**</font>

In [263]:
%env WANDB_PROJECT=hw6_partc_instruction_lmh


env: WANDB_PROJECT=hw6_partc_instruction_lmh


##  <font color = 'indianred'> **6.6 Training**

In [264]:
try:
    # Your code that may cause a CUDA out-of-memory error
    # Example: trainer.train() or other GPU intensive operations
    # lora_model.config.use_cache = False
    trainer.train()
except RuntimeError as e:
    if 'CUDA out of memory' in str(e):
        print("CUDA out of memory error detected. Freeing GPU memory.")
        free_gpu_memory()
        # Optionally, you can retry the operation here after freeing up memory
        # Example retry:
        # trainer.train()
    else:
        raise e


The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'eos_token_id': 1}.


Step,Training Loss,Validation Loss,Entropy,Num Tokens,Mean Token Accuracy
20,1.245300,0.902802,1.809824,82665.000000,0.848121
40,0.302200,0.912452,1.751202,155128.000000,0.849782
60,0.408000,0.690198,1.816243,236470.000000,0.868143
80,0.267000,0.658232,1.812231,310176.000000,0.864570
100,0.372100,0.562868,1.871320,389892.000000,0.871742
120,0.242900,0.562613,1.808682,463890.000000,0.871858
140,0.337700,0.517743,1.882121,542690.000000,0.868743
160,0.251900,0.450124,1.882897,618241.000000,0.879709
180,0.413200,0.425741,1.970209,695252.000000,0.872061
200,0.229100,0.408254,1.988029,772044.000000,0.874822


##  <font color = 'indianred'> **6.6 Export best adapter for Part C inference**</font>


In [265]:
best_model_checkpoint_step = trainer.state.best_model_checkpoint.split('-')[-1]
best_model_checkpoint_step


'460'

In [266]:
best_model_checkpoint_step

'460'

In [267]:
best_ckpt = trainer.state.best_model_checkpoint
print("best_ckpt:", best_ckpt)

final_adapter_dir = project_folder / "models" / "partc_best_adapter_export"
final_adapter_dir.mkdir(parents=True, exist_ok=True)

trainer.model.save_pretrained(final_adapter_dir)
tokenizer.save_pretrained(final_adapter_dir)

print("Saved adapter to:", final_adapter_dir)


best_ckpt: /content/models/gemma_partc_instruction_lmh/checkpoint-460
Saved adapter to: /content/drive/My Drive/Fall_2025_Notes/NLP/HW 6/models/partc_best_adapter_export


In [268]:
wandb.finish()

eval/entropy,▃▂▃▃▅▃▅▅██▆▆▅▄▃▃▃▃▃▂▂▁▁▁▁
eval/loss,██▅▅▄▄▄▃▃▂▂▂▂▂▁▁▁▁▁▁▁▁▁▁▁
eval/mean_token_accuracy,▁▁▃▃▄▄▄▅▄▄▅▅▅▆▇█▇████████
eval/num_tokens,▁▁▂▂▂▂▃▃▃▄▄▄▄▅▅▅▆▆▆▇▇▇▇██
eval/runtime,▂▁▂▂▂█▃▂▁▃▂▁▁▂▂▂▂▃▂▂▂▂▃▂▁
eval/samples_per_second,▇█▇▇▇▁▆▇█▆▇██▇▇▇▇▆▇▇▇▇▆▇█
eval/steps_per_second,▇█▇▇▇▁▆▇█▆▇██▇▇▇▇▆▇▇▇▇▆▇█
train/entropy,▂▂▅▃▆▄▅▅▆█▇▇▄▆▃▅▂▆▁▆▁▅▁▃▁
train/epoch,▁▁▁▁▂▂▂▂▂▂▃▃▃▃▃▄▄▄▄▄▅▅▅▅▅▆▆▆▆▆▆▇▇▇▇▇████
train/global_step,▁▁▁▁▂▂▂▂▂▂▃▃▃▃▃▄▄▄▄▄▅▅▅▅▅▆▆▆▆▆▆▆▇▇▇▇▇███
+5,...
